# Plot the Evolution of Service Territory Maps

This notebook plots the evolution of the map of service territory for a given BA.

In [ ]:
# Start by importing the packages we need:
import os

import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd

from glob import glob
from mpl_toolkits.axes_grid1 import make_axes_locatable


## Set the Directory Structure

In [ ]:
# Identify the top-level directory and the subdirectory where the data will be stored:
ba_mapping_input_dir =  '/Users/burl878/Documents/Code/code_repos/tell/tell/tell_data/tell_quickstarter_data/outputs/ba_service_territory'
population_input_dir = '/Users/burl878/Documents/Code/code_repos/tell/tell/tell_data/tell_raw_data/Population'
county_shapefile_input_dir = '/Users/burl878/Documents/Code/code_repos/tell/tell/tell_data/tell_raw_data/County_Shapefiles'
state_shapefile_input_dir = '/Users/burl878/Documents/Code/code_repos/tell/tell/tell_data/tell_raw_data/State_Shapefiles/'
image_output_dir =  '/Users/burl878/Documents/Code/code_repos/tell/tell/tell_data/visualizations/ba_service_territory'


## Process the Data

In [ ]:
# Define a function process the mapping file over a range of years:
def process_ba_mapping_evolution(ba_mapping_input_dir: str, population_input_dir: str, ba_to_process: str, start_year: int, end_year: int):
    # Loop over the years from the start_year to the end_year and read in the mapping data:
    for year in range(start_year, end_year, 1):
        # Read in the BA-to-county mapping file for that year:
        ba_mapping_df = pd.read_csv((os.path.join(ba_mapping_input_dir, f'ba_service_territory_{str(year)}.csv')), index_col=None, header=0)
            
        # Subset the data to only the BA you want to plot:
        ba_mapping_df = ba_mapping_df.loc[(ba_mapping_df['BA_Code'] == ba_to_process)]

        # Read in county populations file:
        population_df = pd.read_csv(os.path.join(population_input_dir, r'county_populations_2000_to_2023.csv'))

        # Keep only the columns we need:
        population_df = population_df[['county_FIPS', ('pop_' + str(year))]].copy(deep=False)

        # Rename the columns:
        population_df.rename(columns={"county_FIPS": "County_FIPS", ('pop_' + str(year)): "Population"}, inplace=True)

        # Merge the ba_mapping_df and population_df together using county FIPS codes to join them:
        ba_mapping_df = ba_mapping_df.merge(population_df, on='County_FIPS', how='left')
    
        # Concatenate all the years into a single dataframe:
        if year == start_year:
           output_df = ba_mapping_df.copy()
        else:
           output_df = pd.concat([output_df, ba_mapping_df])

    return output_df


In [ ]:
mapping_df = process_ba_mapping_evolution(ba_mapping_input_dir = ba_mapping_input_dir,
                                          population_input_dir = population_input_dir,
                                          ba_to_process = 'CISO',
                                          start_year = 2015,
                                          end_year = 2024)

mapping_df


## Make the Plot

In [ ]:
# Define a function to plot the BA-to-county maps:
def plot_ba_mapping_evolution(ba_mapping_input_dir: str, population_input_dir: str, county_shapefile_input_dir: str, state_shapefile_input_dir: str,
                              ba_to_process: str, start_year: int, end_year: int, image_output_dir: str, image_resolution: int, save_images=False):

    # Process the mapping file using the function defined above:
    mapping_df = process_ba_mapping_evolution(ba_mapping_input_dir = ba_mapping_input_dir,
                                              population_input_dir = population_input_dir,
                                              ba_to_process = ba_to_process,
                                              start_year = start_year,
                                              end_year = end_year)

    # Read in the county shapefile and reassign the 'FIPS' variable as integers:
    counties_df = gpd.read_file(os.path.join(county_shapefile_input_dir, r'tl_2020_us_county.shp')).rename(columns={'GEOID': 'County_FIPS'})
    counties_df['County_FIPS'] = counties_df['County_FIPS'].astype(int)

    # Merge the ba_mapping_df and counties_df together using county FIPS codes to join them:
    counties_df = counties_df.merge(mapping_df, on='County_FIPS', how='left')

    # Subset to only the BA you want to plot:
    counties_subset_df = counties_df.loc[counties_df['BA_Code'] == ba_to_process]
    
    # Read in the state shapefile:
    states_df = gpd.read_file(os.path.join(state_shapefile_input_dir, 'tl_2020_us_state.shp')).rename(columns={'NAME': 'State_Name'})
        
    # Subset the state shapefile to just the states contained in the maping file:
    state_subset_df = states_df[states_df['State_Name'].isin(mapping_df['State_Name'].unique().tolist())]
        
    # Set the colormap and colorlimit:
    colors = plt.get_cmap('YlGnBu', 15)
    pop_max = counties_subset_df['Population'].max()
    
    # Subset based on year:
    counties_a_df = counties_subset_df.loc[counties_subset_df['Year'] == 2016]
    counties_b_df = counties_subset_df.loc[counties_subset_df['Year'] == 2017]
    counties_c_df = counties_subset_df.loc[counties_subset_df['Year'] == 2018]
    counties_d_df = counties_subset_df.loc[counties_subset_df['Year'] == 2019]
    counties_e_df = counties_subset_df.loc[counties_subset_df['Year'] == 2020]
    counties_f_df = counties_subset_df.loc[counties_subset_df['Year'] == 2021]
    counties_g_df = counties_subset_df.loc[counties_subset_df['Year'] == 2022]
    counties_h_df = counties_subset_df.loc[counties_subset_df['Year'] == 2023]
    
    # Create the figure:
    fig, axes = plt.subplots(nrows=2, ncols=4, figsize=(25,13))
    
    # Plot each year using subplots starting in 2016
    ax1 = counties_a_df.plot(ax=axes[0,0], column='Population', cmap=colors, vmin=0, vmax=pop_max, edgecolor='k', linewidth=1)
    state_subset_df.boundary.plot(ax=axes[0,0], linewidth=2, color='k')
    ax1.set_xlabel('Longitude', fontsize=14); 
    ax1.set_ylabel('Latitude', fontsize=14);
    ax1.set_title((ba_to_process + ' Territory: 2016'), fontsize=16)
    ax1.set_title('a)', loc='left', fontsize=16)
    
    ax2 = counties_b_df.plot(ax=axes[0,1], column='Population', cmap=colors, vmin=0, vmax=pop_max, edgecolor='k', linewidth=1)
    state_subset_df.boundary.plot(ax=axes[0,1], linewidth=2, color='k')
    ax2.set_xlabel('Longitude', fontsize=14); 
    ax2.set_ylabel('Latitude', fontsize=14);
    ax2.set_title((ba_to_process + ' Territory: 2017'), fontsize=16)
    ax2.set_title('b)', loc='left', fontsize=16)   

    ax3 = counties_c_df.plot(ax=axes[0,2], column='Population', cmap=colors, vmin=0, vmax=pop_max, edgecolor='k', linewidth=1)
    state_subset_df.boundary.plot(ax=axes[0,2], linewidth=2, color='k')
    ax3.set_xlabel('Longitude', fontsize=14); 
    ax3.set_ylabel('Latitude', fontsize=14);
    ax3.set_title((ba_to_process + ' Territory: 2018'), fontsize=16)
    ax3.set_title('c)', loc='left', fontsize=16)

    ax4 = counties_d_df.plot(ax=axes[0,3], column='Population', cmap=colors, vmin=0, vmax=pop_max, edgecolor='k', linewidth=1)
    state_subset_df.boundary.plot(ax=axes[0,3], linewidth=2, color='k')
    ax4.set_xlabel('Longitude', fontsize=14); 
    ax4.set_ylabel('Latitude', fontsize=14);
    ax4.set_title((ba_to_process + ' Territory: 2019'), fontsize=16)
    ax4.set_title('d)', loc='left', fontsize=16)

    ax5 = counties_e_df.plot(ax=axes[1,0], column='Population', cmap=colors, vmin=0, vmax=pop_max, edgecolor='k', linewidth=1)
    state_subset_df.boundary.plot(ax=axes[1,0], linewidth=2, color='k')
    ax5.set_xlabel('Longitude', fontsize=14); 
    ax5.set_ylabel('Latitude', fontsize=14);
    ax5.set_title((ba_to_process + ' Territory: 2020'), fontsize=16)
    ax5.set_title('e)', loc='left', fontsize=16)

    ax6 = counties_f_df.plot(ax=axes[1,1], column='Population', cmap=colors, vmin=0, vmax=pop_max, edgecolor='k', linewidth=1)
    state_subset_df.boundary.plot(ax=axes[1,1], linewidth=2, color='k')
    ax6.set_xlabel('Longitude', fontsize=14); 
    ax6.set_ylabel('Latitude', fontsize=14);
    ax6.set_title((ba_to_process + ' Territory: 2021'), fontsize=16)
    ax6.set_title('f)', loc='left', fontsize=16)

    ax7 = counties_g_df.plot(ax=axes[1,2], column='Population', cmap=colors, vmin=0, vmax=pop_max, edgecolor='k', linewidth=1)
    state_subset_df.boundary.plot(ax=axes[1,2], linewidth=2, color='k')
    ax7.set_xlabel('Longitude', fontsize=14); 
    ax7.set_ylabel('Latitude', fontsize=14);
    ax7.set_title((ba_to_process + ' Territory: 2022'), fontsize=16)
    ax7.set_title('g)', loc='left', fontsize=16)

    ax8 = counties_h_df.plot(ax=axes[1,3], column='Population', cmap=colors, vmin=0, vmax=pop_max, edgecolor='k', linewidth=1)
    state_subset_df.boundary.plot(ax=axes[1,3], linewidth=2, color='k')
    ax8.set_xlabel('Longitude', fontsize=14); 
    ax8.set_ylabel('Latitude', fontsize=14);
    ax8.set_title((ba_to_process + ' Territory: 2023'), fontsize=16)
    ax8.set_title('h)', loc='left', fontsize=16)
    
    # If the "save_images" flag is set to true then save the plot to a .png file:
    if save_images == True:
       filename = ('BA_Service_Territory_Evolution_' + ba_to_process + '.png')
       plt.savefig(os.path.join(image_output_dir, filename), dpi=image_resolution, bbox_inches='tight', facecolor='white')
    

In [ ]:
plot_ba_mapping_evolution(ba_mapping_input_dir = ba_mapping_input_dir,
                          population_input_dir = population_input_dir,
                          county_shapefile_input_dir = county_shapefile_input_dir,
                          state_shapefile_input_dir = state_shapefile_input_dir,
                          ba_to_process = 'NEVP',
                          start_year = 2015,
                          end_year = 2024,
                          image_output_dir = image_output_dir,
                          image_resolution = 300,
                          save_images = True)
